# Band-edge discriminant gain and near-lock slope

This notebook is the slower companion to `notes/band-edge-discriminant-gain-and-slope.md`.

The narrow question is the useful one: **does the raw band-edge imbalance panel measure the same thing as the normalized near-lock slope?**


## The local setup

- SRRC QPSK
- `4 samples/symbol`
- `1024` symbols
- central finite difference at `±0.01 R_s`
- tap counts `{63, 127, 255}`
- roll-off `{0.05, 0.20, 0.35, 0.50}`


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

rows = []
with (Path('..') / 'assets' / '2026-05-20-band-edge-discriminant-slope-check.csv').open() as handle:
    for row in csv.DictReader(handle):
        parsed = {}
        for key, value in row.items():
            if key in {'samples_per_symbol', 'symbol_count', 'seed', 'trim', 'tap_count'}:
                parsed[key] = int(value)
            else:
                parsed[key] = float(value)
        rows.append(parsed)

len(rows), rows[0].keys()


## One table is enough to see the point

The 255-tap rows are the clearest summary because they get closest to the normalized target slope of `1`.


In [ ]:
for row in rows:
    if row["tap_count"] == 255:
        print(
            f"alpha={row['rolloff']:.2f}  raw@0.10={row['imbalance_at_0p10']:.3f}  slope={row['central_slope_wrt_deltaf_over_Rs']:.3f}"
        )


## Group by tap count

This makes the second story visible: short filters understate the near-lock slope, especially when roll-off is small.


In [ ]:
grouped = defaultdict(list)
for row in rows:
    grouped[row["tap_count"]].append(row)

for tap_count, series in sorted(grouped.items()):
    print(f"\n{tap_count} taps")
    for row in sorted(series, key=lambda entry: entry["rolloff"]):
        print(
            f"  alpha={row['rolloff']:.2f}  slope={row['central_slope_wrt_deltaf_over_Rs']:.3f}  raw@0.10={row['imbalance_at_0p10']:.3f}"
        )


## A compact ratio check

The raw imbalance changes much more across roll-off than the calibrated slope does once the filters are long enough.


In [ ]:
rows_255 = {row["rolloff"]: row for row in rows if row["tap_count"] == 255}
raw_ratio = rows_255[0.50]["imbalance_at_0p10"] / rows_255[0.05]["imbalance_at_0p10"]
slope_span = rows_255[0.50]["central_slope_wrt_deltaf_over_Rs"] - rows_255[0.20]["central_slope_wrt_deltaf_over_Rs"]
raw_ratio, slope_span


## Takeaway

- keep the old raw-imbalance panel as the intuition view
- use the near-zero slope when the question is normalized gain near lock
- treat tiny roll-off as both a waveform problem and a filter-design problem
